In [1]:
import pandas as pd
import os

In [2]:
def load_and_prep_nutrition_matrix(data_dir):
    print("Iniciando pipeline de carga de datos FDC con Micronutrientes...")
    
    # Cargar CSVs
    df_food = pd.read_csv(os.path.join(data_dir, 'food.csv'), usecols=['fdc_id', 'description'])
    df_nutrient = pd.read_csv(os.path.join(data_dir, 'nutrient.csv'), usecols=['id', 'name', 'unit_name'])
    df_food_nutrient = pd.read_csv(os.path.join(data_dir, 'food_nutrient.csv'), usecols=['fdc_id', 'nutrient_id', 'amount'])
    df_portion = pd.read_csv(os.path.join(data_dir, 'food_portion.csv'), usecols=['fdc_id', 'amount', 'measure_unit_id', 'modifier', 'gram_weight'])
    df_measure_unit = pd.read_csv(os.path.join(data_dir, 'measure_unit.csv'), usecols=['id', 'name'])

    # IDs de Nutrientes Ampliados:
    # Macros: 1008 (Energía), 1003 (Proteína), 1004 (Grasas Tot.), 1005 (Carbos)
    # Salud: 1093 (Sodio), 1253 (Colesterol), 1258 (Grasas Saturadas), 1079 (Fibra), 1063 (Azúcar)
    target_nutrients = [1008, 1003, 1004, 1005, 1093, 1253, 1258, 1079, 1063] 
    critical_macros = [1008, 1003, 1004, 1005] # Solo requeridos estrictamente
    
    df_fn_filtered = df_food_nutrient[df_food_nutrient['nutrient_id'].isin(target_nutrients)].copy()
    df_fn_filtered = df_fn_filtered.drop_duplicates(subset=['fdc_id', 'nutrient_id'])

    # Pivotear matriz
    nutrition_matrix = df_fn_filtered.pivot(index='fdc_id', columns='nutrient_id', values='amount').reset_index()
    nutrient_names = df_nutrient.set_index('id')['name'].to_dict()
    nutrition_matrix = nutrition_matrix.rename(columns=nutrient_names)

    final_df = pd.merge(df_food, nutrition_matrix, on='fdc_id', how='inner')

    # Porciones
    df_portion_unique = df_portion.drop_duplicates(subset=['fdc_id']).copy()
    df_portion_unique = pd.merge(df_portion_unique, df_measure_unit, left_on='measure_unit_id', right_on='id', how='left')
    final_df = pd.merge(final_df, df_portion_unique[['fdc_id', 'amount', 'name', 'modifier', 'gram_weight']], on='fdc_id', how='left')
    final_df = final_df.rename(columns={'amount': 'portion_amount', 'name': 'portion_unit'})

    # LIMPIEZA INTELIGENTE
    macro_cols = [nutrient_names[id] for id in critical_macros if id in nutrient_names]
    micro_cols = [nutrient_names[id] for id in target_nutrients if id not in critical_macros and id in nutrient_names]
    
    # 1. Drop a los que no tienen Macros
    final_df = final_df.dropna(subset=macro_cols)
    
    # 2. Rellenar con 0 los micronutrientes faltantes (ej. pollo no tiene fibra, le ponemos 0)
    for col in micro_cols:
        if col in final_df.columns:
            final_df[col] = final_df[col].fillna(0.0)
    
    final_df['gram_weight'] = final_df['gram_weight'].fillna(100.0)
    final_df['portion_unit'] = final_df['portion_unit'].fillna('g')
    final_df['portion_amount'] = final_df['portion_amount'].fillna(100.0)

    # ESCALADO DE NUTRIENTES (De 100g a Porción Real)
    all_nutrients = macro_cols + micro_cols
    for col in all_nutrients:
        if col in final_df.columns:
            final_df[col] = (final_df[col] * final_df['gram_weight']) / 100.0

    print(f"Pipeline completado. Matriz final lista para optimizar.\n")
    return final_df, nutrient_names

In [3]:
import pulp

In [4]:
def calcular_requerimientos():
    print("--- 📊 Perfil Nutricional ---")
    # Captura de datos del usuario
    peso = float(input("Peso en kg (ej. 75): "))
    altura = float(input("Altura en cm (ej. 175): "))
    edad = int(input("Edad (ej. 26): ") or 26) # Por defecto 26 si se presiona Enter
    sexo = input("Sexo (M/F): ").strip().upper()
    
    print("\nNiveles de actividad física:")
    print("1. Sedentario (Poco o nada de ejercicio)")
    print("2. Ligero (Ejercicio ligero 1-3 días/semana)")
    print("3. Moderado (Ejercicio moderado 3-5 días/semana)")
    print("4. Activo (Ejercicio fuerte 6-7 días/semana)")
    print("5. Muy Activo (Ejercicio extremo o trabajo físico)")
    actividad_opcion = input("Selecciona tu nivel (1-5): ")
    
    factores = {'1': 1.2, '2': 1.375, '3': 1.55, '4': 1.725, '5': 1.9}
    factor_actividad = factores.get(actividad_opcion, 1.2)

    # 1. Tasa Metabólica Basal (BMR) usando Mifflin-St Jeor
    if sexo == 'M':
        bmr = (10 * peso) + (6.25 * altura) - (5 * edad) + 5
    else:
        bmr = (10 * peso) + (6.25 * altura) - (5 * edad) - 161
        
    # 2. Gasto Energético Total (Límite Calórico)
    tdee = bmr * factor_actividad
    
    max_lipidos = (tdee * 0.30) / 9
    max_carbos = (tdee * 0.45) / 4
    max_saturadas = (tdee * 0.10) / 9  # OMS: <10% de calorías de grasas saturadas
    
    return tdee, max_lipidos, max_carbos, max_saturadas

In [5]:
def optimizar_dieta_saludable(df, nut_names, max_cal, max_grasa, max_carbos, max_sat):
    print("Resolviendo Modelo MILP Clínico...\n")
    
    alimentos = df['description'].tolist()
    
    # Diccionarios de acceso rápido
    energia = dict(zip(alimentos, df[nut_names[1008]]))
    proteina = dict(zip(alimentos, df[nut_names[1003]]))
    grasa = dict(zip(alimentos, df[nut_names[1004]]))
    carbos = dict(zip(alimentos, df[nut_names[1005]]))
    
    sodio = dict(zip(alimentos, df.get(nut_names[1093], pd.Series([0]*len(df)))))
    colesterol = dict(zip(alimentos, df.get(nut_names[1253], pd.Series([0]*len(df)))))
    grasas_sat = dict(zip(alimentos, df.get(nut_names[1258], pd.Series([0]*len(df)))))
    fibra = dict(zip(alimentos, df.get(nut_names[1079], pd.Series([0]*len(df)))))
    azucar = dict(zip(alimentos, df.get(nut_names[1063], pd.Series([0]*len(df)))))
    
    prob = pulp.LpProblem("Dieta_Saludable_Max_Proteina", pulp.LpMaximize)
    
    # Variables de Decisión Enteras
    x = pulp.LpVariable.dicts("Porciones", alimentos, lowBound=0, cat='Integer')
    
    # 🎯 Función Objetivo: Maximizar Proteína
    prob += pulp.lpSum([proteina[i] * x[i] for i in alimentos]), "Total_Proteina"
    
    # 🛑 RESTRICCIONES MACRONUTRIENTES
    prob += pulp.lpSum([energia[i] * x[i] for i in alimentos]) <= max_cal, "Max_Calorias"
    prob += pulp.lpSum([grasa[i] * x[i] for i in alimentos]) <= max_grasa, "Max_Grasa"
    prob += pulp.lpSum([carbos[i] * x[i] for i in alimentos]) <= max_carbos, "Max_Carbohidratos"
    
    # 🏥 RESTRICCIONES DE SALUD Y MICRONUTRIENTES (OMS/FDA)
    prob += pulp.lpSum([sodio[i] * x[i] for i in alimentos]) <= 2300, "Max_Sodio_mg"
    prob += pulp.lpSum([colesterol[i] * x[i] for i in alimentos]) <= 300, "Max_Colesterol_mg"
    prob += pulp.lpSum([grasas_sat[i] * x[i] for i in alimentos]) <= max_sat, "Max_Saturadas"
    prob += pulp.lpSum([fibra[i] * x[i] for i in alimentos]) >= 25, "Min_Fibra_g"
    prob += pulp.lpSum([azucar[i] * x[i] for i in alimentos]) <= 50, "Max_Azucar_g"
    
# 🥗 LÍMITES POR ALIMENTO INDIVIDUAL (Fisiología y Variedad)
    # Usamos enumerate() para obtener un índice (idx) único para cada iteración
    for idx, i in enumerate(alimentos):
        peso_porcion = df.loc[df['description'] == i, 'gram_weight'].values[0]
        
        # Criterio de Capacidad: Maximo ~400g del mismo alimento
        peso_limite = max(400.0, peso_porcion)
        # Bautizamos la restricción usando el índice para garantizar que sea única
        prob += (x[i] * peso_porcion) <= peso_limite, f"Volumen_Max_{idx}"
        
        # Criterio de Variedad: Un alimento no puede darte más del 25% de tus calorías
        prob += (energia[i] * x[i]) <= (max_cal * 0.25), f"Energia_Max_{idx}"

    # Resolver
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Resultados
    if prob.status == pulp.LpStatusOptimal:
        print("✅ ¡DIETA FACTIBLE ENCONTRADA!\n")
        print(f"🥩 Proteína Maximizada: {pulp.value(prob.objective):.1f} g")
        print(f"🔥 Calorías: {sum([energia[i]*x[i].varValue for i in alimentos]):.0f} / {max_cal:.0f} kcal")
        print(f"🌾 Fibra lograda: {sum([fibra[i]*x[i].varValue for i in alimentos]):.1f} g (Min 25g)")
        print(f"🧂 Sodio consumido: {sum([sodio[i]*x[i].varValue for i in alimentos]):.0f} mg (Max 2300mg)")
        
        print("\n🛒 MENÚ RECOMENDADO:")
        for i in alimentos:
            if x[i].varValue > 0:
                unid = df.loc[df['description'] == i, 'portion_unit'].values[0]
                cant = df.loc[df['description'] == i, 'portion_amount'].values[0]
                print(f"- {int(x[i].varValue)} porciones de '{i}' ({x[i].varValue * cant:.1f} {unid})")
    else:
        print("❌ Solución Infactible.")
        print("El algoritmo no pudo encontrar una dieta que cumpla todas las reglas a la vez.")
        print("Sugerencia: Reduce el mínimo de fibra o relaja el tope de sodio.")

In [ ]:
# --- EJECUCIÓN ---
if __name__ == "__main__":
    # Ajusta tu ruta aquí
    directorio_datos = r'C:\Users\PC RST\Downloads\trabajo_publico\trabajo_publico\proyecto alimenticio'
    df_matriz, dict_nombres = load_and_prep_nutrition_matrix(directorio_datos)
    
    tdee, max_lip, max_carb, max_sat = calcular_requerimientos()
    optimizar_dieta_saludable(df_matriz, dict_nombres, tdee, max_lip, max_carb, max_sat)

Iniciando pipeline de carga de datos FDC con Micronutrientes...
Pipeline completado. Matriz final lista para optimizar.

--- 📊 Perfil Nutricional ---

Niveles de actividad física:
1. Sedentario (Poco o nada de ejercicio)
2. Ligero (Ejercicio ligero 1-3 días/semana)
3. Moderado (Ejercicio moderado 3-5 días/semana)
4. Activo (Ejercicio fuerte 6-7 días/semana)
5. Muy Activo (Ejercicio extremo o trabajo físico)
Resolviendo Modelo MILP Clínico...

✅ ¡DIETA FACTIBLE ENCONTRADA!

🥩 Proteína Maximizada: 353.1 g
🔥 Calorías: 2177 / 2195 kcal
🌾 Fibra lograda: 25.5 g (Min 25g)
🧂 Sodio consumido: 2300 mg (Max 2300mg)

🛒 MENÚ RECOMENDADO:
- 1 porciones de 'Broccoli, raw' (1.0 cup)
- 14 porciones de 'Egg, white, raw, frozen, pasteurized' (14.0 oz)
- 13 porciones de 'Egg, white, dried' (13.0 tablespoon)
- 3 porciones de 'Kale, frozen, cooked, boiled, drained, without salt' (3.0 cup)
- 1 porciones de 'Lettuce, cos or romaine, raw' (1.0 bunch)
- 2 porciones de 'Yogurt, Greek, plain, nonfat' (2.0 contain